# Paperless-ngx ML Platform — Chameleon Provisioning

Solo build. Brings up the full integrated ML system on a single VM:

- Paperless-ngx (custom fork with HTR + semantic-search UI)
- Data stack: PostgreSQL × 2, MinIO, Redpanda, Qdrant, Adminer
- ML serving: **ml_gateway** (pretrained TrOCR + mpnet)
- **qdrant_indexer** (chunks merged_text, populates Qdrant)
- **drift_monitor** (online MMD against IAM reference)
- **htr_consumer** (Kafka→slice→HTR→DB)
- Observability: Prometheus, Grafana (pre-provisioned dashboards), Alertmanager, MLflow, rollback controller

**Pre-reqs on your local machine before running this notebook:**
- `ghcr.io/redes01/paperless-ngx-ml:latest` has been pushed to GHCR (via `scripts/build_and_push.ps1`). If you've already pushed once and haven't changed the fork, skip.
- `paperless_data` and `paperless_data_integration` repos on GitHub are up to date.

Running this notebook provisions the VM, installs Docker, clones both repos, builds every service image on the VM, and starts everything. Expect **30–45 minutes total bring-up** on first run (most of which is pip installing torch + transformers for ml_gateway and qdrant_indexer).

---
## Part 1 — VM Setup

### Step 1 — Select project and site

In [ ]:
from chi import server, context, lease, network
import chi, os, datetime, time

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")
username = os.getenv("USER")
print(f"Username: {username}")

### Step 2 — Reserve VM

Flavor: `m1.xlarge` (8 vCPU / 16GB RAM). ml_gateway loads TrOCR (~650MB) + mpnet (~450MB) into RAM, drift_monitor loads its own small CNN, Prometheus + Grafana + MLflow are light. 16GB leaves headroom.

In [ ]:
l = lease.Lease(
    f"lease-paperless-integration-{username}",
    duration=datetime.timedelta(hours=12),
)
l.add_flavor_reservation(id=chi.server.get_flavor_id("m1.xlarge"), amount=1)
l.submit(idempotent=True)
l.show()

### Step 3 — Launch VM

In [ ]:
s = server.Server(
    f"node-paperless-integration-{username}",
    image_name="CC-Ubuntu24.04",
    flavor_name=l.get_reserved_flavors()[0].name,
)
s.submit(idempotent=True)

### Step 4 — Assign floating IP

In [ ]:
s.associate_floating_ip()
s.refresh()
s.show(type="widget")

### Step 5 — Open security groups

Ports needed:
- `22`   SSH
- `8000` Paperless UI
- `9001` MinIO console
- `5050` Adminer
- `8090` Redpanda console
- `6333` Qdrant dashboard
- `3000` Grafana
- `9090` Prometheus
- `9093` Alertmanager
- `5000` MLflow UI

In [ ]:
security_groups = [
    {"name": "allow-ssh",  "port": 22,   "description": "SSH"},
    {"name": "allow-8000", "port": 8000, "description": "Paperless UI"},
    {"name": "allow-5050", "port": 5050, "description": "Adminer"},
    {"name": "allow-9001", "port": 9001, "description": "MinIO console"},
    {"name": "allow-8090", "port": 8090, "description": "Redpanda console"},
    {"name": "allow-6333", "port": 6333, "description": "Qdrant"},
    {"name": "allow-3000", "port": 3000, "description": "Grafana"},
    {"name": "allow-9090", "port": 9090, "description": "Prometheus"},
    {"name": "allow-9093", "port": 9093, "description": "Alertmanager"},
    {"name": "allow-5000", "port": 5000, "description": "MLflow"},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({"name": sg["name"], "description": sg["description"]})
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])

print(f"Applied {len(security_groups)} security groups")

### Step 6 — Install Docker

In [ ]:
s.refresh()
s.check_connectivity()
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")
print("Docker installed.")

---
## Part 2 — Deploy the stack

### Step 7 — Clone repos

Both from `REDES01`. `paperless_data` contains data pipelines + quality scripts + drift reference builder; `paperless_data_integration` contains every service compose.

In [ ]:
DATA_REPO        = "https://github.com/REDES01/paperless_data.git"
INTEGRATION_REPO = "https://github.com/REDES01/paperless_data_integration.git"

s.execute("rm -rf ~/paperless_data ~/paperless_data_integration")
s.execute(f"git clone {DATA_REPO} ~/paperless_data")
s.execute(f"git clone {INTEGRATION_REPO} ~/paperless_data_integration")
s.execute("ls ~/")
print("Repos cloned.")

### Step 8 — Write Paperless secret key

Must run this before bringing up the Paperless stack.

In [ ]:
s.execute(
    "cd ~/paperless_data_integration/paperless && "
    "cp docker-compose.env.example docker-compose.env && "
    "SECRET=$(python3 -c 'import secrets; print(secrets.token_urlsafe(64))') && "
    "sed -i \"s|PAPERLESS_SECRET_KEY=replace-me-with-a-real-secret|PAPERLESS_SECRET_KEY=$SECRET|\" docker-compose.env && "
    "grep PAPERLESS_SECRET_KEY docker-compose.env"
)
print("Secret key written.")

### Step 9 — Pull Paperless image from GHCR

In [ ]:
s.execute("sg docker -c 'docker pull ghcr.io/redes01/paperless-ngx-ml:latest'")
print("Paperless image pulled.")

### Step 10 — Bring up data stack + apply ML schema migration

Starts PostgreSQL (data-stack), MinIO, Redpanda, Qdrant, Adminer. Then applies the `paperless_doc_id` migration idempotently so the HTR consumer's upserts work.

In [ ]:
s.execute("cd ~/paperless_data_integration && sg docker -c 'bash scripts/create_network.sh'")
s.execute("cd ~/paperless_data_integration && sg docker -c 'bash scripts/up_paperless_data.sh'")

print("Waiting 25s for postgres + minio to become healthy...")
time.sleep(25)

# paperless_doc_id migration (idempotent)
s.execute(
    "cat ~/paperless_data_integration/seed/phase2_add_paperless_doc_id.sql | "
    "sg docker -c 'docker exec -i postgres psql -U user -d paperless'"
)
print("Data stack up, ML schema migrated.")

### Step 11 — Bring up Paperless

In [ ]:
s.execute("cd ~/paperless_data_integration && sg docker -c 'bash scripts/up_paperless.sh'")
print("Waiting 45s for Paperless to finish starting...")
time.sleep(45)
s.execute("sg docker -c 'docker ps --filter name=paperless --format \"{{.Names}}\t{{.Status}}\"'")

### Step 12 — Create Paperless superuser + generate API token

In [ ]:
# Create superuser (admin/admin)
s.execute(
    "sg docker -c 'docker exec paperless-webserver-1 python manage.py shell -c \""
    "from django.contrib.auth.models import User; "
    "User.objects.filter(username=\\\"admin\\\").exists() or "
    "User.objects.create_superuser(\\\"admin\\\", \\\"admin@example.com\\\", \\\"admin\\\"); "
    "print(\\\"Superuser ready\\\")\"'"
)

# Fetch API token for admin (NOT User.objects.first(), which returns AnonymousUser)
result = s.execute(
    "sg docker -c 'docker exec paperless-webserver-1 python manage.py shell -c \""
    "from rest_framework.authtoken.models import Token; "
    "from django.contrib.auth.models import User; "
    "t, _ = Token.objects.get_or_create(user=User.objects.get(username=\\\"admin\\\")); "
    "print(t.key)\"'"
)
PAPERLESS_TOKEN = result.stdout.strip().split("\n")[-1]
print(f"API Token: {PAPERLESS_TOKEN}")

### Step 13 — Build and start all ML services

This step builds images on the VM for:
- `ml_gateway` (TrOCR + mpnet) — **longest, ~15 min** on first build (pip install torch + transformers + pre-download models)
- `qdrant_indexer` (mpnet cached from earlier build)
- `drift_monitor`
- `htr_consumer`
- `observability` stack (prometheus, alertmanager, grafana, mlflow, rollback_ctrl)

All containers join the shared `paperless_ml_net`. Grafana dashboards (drift + ml_gateway) are pre-provisioned from `observability/grafana/dashboards/*.json` — no manual import needed.

In [ ]:
print("Building and starting ml_gateway, qdrant_indexer, drift_monitor, observability, htr_consumer...")
print("First build is ~15 min (downloads + caches TrOCR + mpnet into ml_gateway image)")
print()

# up_all.sh orchestrates all the compose ups in dependency order
s.execute(
    f"cd ~/paperless_data_integration && "
    f"PAPERLESS_TOKEN={PAPERLESS_TOKEN} "
    f"sg docker -c 'bash scripts/up_all.sh'"
)

### Step 14 — Wait for ml_gateway to become healthy

First boot loads TrOCR + mpnet into RAM (~60s). Also give Paperless + other services time to finish warming up.

In [ ]:
print("Waiting 90s for ml_gateway startup + model load...")
time.sleep(90)

s.execute("sg docker -c 'docker ps --format \"table {{.Names}}\t{{.Status}}\"'")
print()
print("ml_gateway health check:")
s.execute("curl -sf http://localhost:8100/health || echo 'not ready yet'")

---
## Part 3 — Smoke tests

### Step 15 — Upload sample documents

Uploads the two committed samples. Paperless ingestion is async (Tesseract OCR + thumbnail); wait ~60s before looking up IDs.

In [ ]:
s.execute(
    f"for f in ~/paperless_data_integration/sample_documents/sample_budget_memo.pdf "
    f"~/paperless_data_integration/sample_documents/sample_scan.jpeg; do "
    f"echo \"Uploading $f...\"; "
    f"curl -s -X POST http://localhost:8000/api/documents/post_document/ "
    f"-H \"Authorization: Token {PAPERLESS_TOKEN}\" "
    f"-F \"document=@$f\"; "
    f"echo; done"
)
print("Uploads submitted. Waiting 60s for Paperless to ingest...")
time.sleep(60)

### Step 16 — Verify the pipeline processed the uploads

Expected:
- htr_consumer logs show each `paperless_doc_id=N processed`
- ML Postgres has rows in `documents` with `paperless_doc_id` populated
- qdrant_indexer logs show `paperless_doc_id=N indexed`
- Qdrant collection `document_chunks` has non-zero points

In [ ]:
print("Waiting 60s for htr_consumer + qdrant_indexer to finish processing both uploads...")
time.sleep(60)

print("\n── htr_consumer logs (last 30) ──")
s.execute("sg docker -c 'docker logs htr_consumer --tail 30'")

print("\n── qdrant_indexer logs (last 20) ──")
s.execute("sg docker -c 'docker logs qdrant_indexer --tail 20'")

print("\n── ML Postgres summary ──")
s.execute(
    "sg docker -c 'docker exec postgres psql -U user -d paperless -c \""
    "SELECT paperless_doc_id, LEFT(merged_text, 60) AS preview, "
    "LENGTH(merged_text) AS chars "
    "FROM documents WHERE paperless_doc_id IS NOT NULL ORDER BY uploaded_at DESC;\"'"
)

print("\n── Qdrant collection stats ──")
s.execute("curl -s http://localhost:6333/collections/document_chunks | python3 -m json.tool")

### Step 17 — Test HTR inference directly

In [ ]:
# Sanity-check ml_gateway by asking it to transcribe any one uploaded crop.
# We pull the first crop URL from the data-stack Postgres.
result = s.execute(
    "sg docker -c 'docker exec postgres psql -U user -d paperless -tAc \""
    "SELECT crop_s3_url FROM handwritten_regions LIMIT 1;\"'"
)
lines = [l for l in result.stdout.strip().split("\n") if l.startswith("s3://")]
if lines:
    url = lines[0]
    print(f"Testing HTR on: {url}")
    s.execute(
        f"curl -s -X POST http://localhost:8100/predict/htr "
        f"-H 'Content-Type: application/json' "
        f'-d \'{{"document_id":"x","page_id":"x","region_id":"x","crop_s3_url":"{url}"}}\' '
        f"| python3 -m json.tool"
    )
else:
    print("No regions in DB yet; upload must still be processing or no handwriting detected.")

### Step 18 — Test semantic search

In [ ]:
s.execute(
    "curl -s -X POST http://localhost:8100/predict/search "
    "-H 'Content-Type: application/json' "
    '-d \'{"query":"budget approval memo","k":3}\' '
    "| python3 -m json.tool"
)

---
## Part 4 — Drift reference + data quality artifacts

This part produces the artifacts that the drift dashboard and training-set manifest reference in the demo video.

### Step 19 — Ingest IAM (needed by drift reference + training baseline)

This populates `s3://paperless-datalake/warehouse/iam_dataset/` with Parquet shards. Takes a couple minutes.

### Step 19b — Ingest SQuAD 2.0

Populates `s3://paperless-datalake/warehouse/squad_dataset/` with Parquet shards for the retrieval model's external dataset. Required so the ingestion validator's SQuAD checks pass.

In [ ]:
s.execute(
    "cd ~/paperless_data && "
    "sg docker -c 'docker run --rm --network paperless_ml_net "
    "-v $PWD:/app -w /app "
    "-e MINIO_ENDPOINT=minio:9000 "
    "-e MINIO_ACCESS_KEY=admin "
    "-e MINIO_SECRET_KEY=paperless_minio "
    "python:3.12-slim bash -c \""
    "pip install -q pyarrow minio tqdm && "
    "python ingestion/ingest_squad.py\"'"
)
print("SQuAD ingestion complete.")

In [ ]:
s.execute(
    "cd ~/paperless_data && "
    "sg docker -c 'docker run --rm --network paperless_ml_net "
    "-v $PWD:/app -w /app "
    "-e MINIO_ENDPOINT=minio:9000 "
    "-e MINIO_ACCESS_KEY=admin "
    "-e MINIO_SECRET_KEY=paperless_minio "
    "python:3.12-slim bash -c \""
    "pip install -q pyarrow minio datasets tqdm Pillow && "
    "python ingestion/ingest_iam.py\"'"
)
print("IAM ingestion complete.")

### Step 20 — Run ingestion validation (Point 1)

Produces `s3://paperless-datalake/warehouse/iam_dataset/_validation/<ts>.json`.

In [ ]:
s.execute(
    "cd ~/paperless_data && "
    "sg docker -c 'docker run --rm --network paperless_ml_net "
    "-v $PWD:/app -w /app "
    "-e MINIO_ENDPOINT=minio:9000 "
    "-e MINIO_ACCESS_KEY=admin "
    "-e MINIO_SECRET_KEY=paperless_minio "
    "python:3.12-slim bash -c \""
    "pip install -q -r batch_pipeline/requirements.txt && "
    "python batch_pipeline/validate_ingestion.py\"'"
)

### Step 21 — Build the drift reference

Fits `MMDDriftOnline` on 500 IAM crops and saves to `s3://paperless-datalake/warehouse/drift_reference/htr_v1/cd/`. drift_monitor's background retry loop picks up the detector automatically within 60 seconds — no restart needed.

In [ ]:
s.execute(
    "cd ~/paperless_data && "
    "sg docker -c 'docker run --rm --network paperless_ml_net "
    "-v $PWD:/app -w /app "
    "-e MINIO_ENDPOINT=minio:9000 "
    "-e MINIO_ACCESS_KEY=admin "
    "-e MINIO_SECRET_KEY=paperless_minio "
    "python:3.12-slim bash -c \""
    "pip install --no-cache-dir --index-url https://download.pytorch.org/whl/cpu torch==2.4.1 && pip install --no-cache-dir alibi-detect pyarrow minio Pillow && "
    "python scripts/build_drift_reference.py\"'"
)


print("Reference detector uploaded to MinIO.")
print("drift_monitor's background retry loop will load it within 60 seconds.")


### Step 22 — Upload OOD samples for the drift demo

In [ ]:
s.execute(
    "cd ~/paperless_data/scripts && "
    "sg docker -c 'docker run --rm --network paperless_ml_net "
    "-v $PWD:/app -w /app "
    "python:3.12-slim bash -c \""
    "pip install -q faker Pillow minio && "
    "python make_ood_samples.py && "
    "MINIO_ENDPOINT=minio:9000 MINIO_ACCESS_KEY=admin MINIO_SECRET_KEY=paperless_minio "
    "python upload_ood_to_minio.py\"'"
)

---
## Part 5 — Access URLs + API token

In [ ]:
s.refresh()
floating_ip = None
for net, addrs in (s.addresses or {}).items():
    for addr in addrs:
        if addr.get("OS-EXT-IPS:type") == "floating":
            floating_ip = addr["addr"]

print(f"Floating IP: {floating_ip}")
print()
print("═" * 70)
print("  Service URLs")
print("═" * 70)
print(f"  Paperless UI        http://{floating_ip}:8000           admin / admin")
print(f"  HTR review          http://{floating_ip}:8000/ml/htr-review")
print(f"  Semantic search     http://{floating_ip}:8000/ml/search")
print(f"  Grafana             http://{floating_ip}:3000           admin / admin")
print(f"  Prometheus          http://{floating_ip}:9090")
print(f"  Alertmanager        http://{floating_ip}:9093")
print(f"  MLflow              http://{floating_ip}:5000")
print(f"  MinIO Console       http://{floating_ip}:9001           admin / paperless_minio")
print(f"  Adminer             http://{floating_ip}:5050           user / paperless_postgres")
print(f"  Redpanda Console    http://{floating_ip}:8090")
print(f"  Qdrant Dashboard    http://{floating_ip}:6333/dashboard")
print()
print(f"  API Token: {PAPERLESS_TOKEN}")
print(f"  SSH:       ssh -i ~/.ssh/id_rsa_chameleon cc@{floating_ip}")

---
## Teardown

Uncomment and run when you're done to release resources.

In [ ]:
# s = server.get_server(f"node-paperless-integration-{username}")
# server.delete_server(s.id)
# l = lease.get_lease(f"lease-paperless-integration-{username}")
# lease.delete_lease(l.id)
# print("Resources released.")